# Invite reviewers to Pretalx

Invites the emails from the reviewer form to the Pretalx team and fills the status columns in the form responses tab.
Run it every 2 to 3 days while the form is open, because people need time to accept the invite.

1. Every form email that is neither a team member nor invited yet gets a Pretalx invite. Pretalx sends the mail.
2. `Recieved welcome mail` gets `TRUE` once someone is invited.
3. `Activated in Pretalx`, `Pretalx Mail` and `Pretalx Name` are filled once someone has accepted.
4. Only empty cells are written. `All Proposals` and `PC Comment` stay with the program committee.
5. Members who accepted with another email than in the form are listed at the end. Fill their `Pretalx Mail` and `Pretalx Name` by hand.

The matching rules are in `reviewer_invites.py`.
The Pretalx token in `~/.pytanis/config.toml` must be valid for all events of the organiser, otherwise the teams API returns 403.

In [ ]:
import tomli
from gspread.utils import rowcol_to_a1

from pytanis import GSheetsClient, PretalxClient
from reviewer_invites import EXTRA_COLUMNS, cell_updates, emails_to_invite, unmatched_members

In [ ]:
# Event-specific settings, see README.md
with open('config.toml', 'rb') as fh:
    cfg = tomli.load(fh)

ORGANISER = 'pyconde'
TEAM_NAME = '2027-Reviewers-ALL'
# True only prints what would happen. Set it to False to send the invites and write the sheet.
DRY_RUN = True

## Read the form responses

In [ ]:
gsheet_client = GSheetsClient(read_only=False)
worksheet = gsheet_client.gsheet(cfg['reviewer_spread_id'], cfg['reviewer_work_name'])
header, *data = worksheet.get_all_values()
missing = [col for col in EXTRA_COLUMNS if col not in header]
header += missing
rows = [dict(zip(header, values + [''] * (len(header) - len(values)))) for values in data]
print(f'{len(rows)} form rows, missing headers: {missing or "none"}')

## Invite new reviewers

In [ ]:
pretalx_client = PretalxClient()
team = pretalx_client.team(ORGANISER, TEAM_NAME)
to_invite = emails_to_invite(rows, team)
print(f'{len(team["members"])} members, {len(team["invites"])} open invites, {len(to_invite)} to invite:')
print('\n'.join(to_invite))

In [ ]:
if DRY_RUN:
    # pretend the invites went out, so the planned cell updates below are complete
    team = {**team, 'invites': team['invites'] + [{'email': email} for email in to_invite]}
else:
    for email in to_invite:
        pretalx_client.invite_to_team(ORGANISER, team['id'], email)
        print(f'invited {email}')
    # read the team again, so the sheet shows what Pretalx really has
    team = pretalx_client.team(ORGANISER, TEAM_NAME)

## Update the status columns

In [ ]:
updates = cell_updates(rows, team)
for pos, col, val in updates:
    print(f'row {pos + 2}: {col} = {val}')

unmatched = unmatched_members(rows, team)
print(f'\n{len(unmatched)} members without a form row. Fill their Pretalx Mail and Pretalx Name by hand:')
for member in unmatched:
    print(f'  {member["name"]} <{member["email"]}>')

In [ ]:
if DRY_RUN:
    print(f'DRY_RUN: would add {len(missing)} headers and write {len(updates)} cells')
else:
    if missing:
        if len(header) > worksheet.col_count:
            worksheet.add_cols(len(header) - worksheet.col_count)
        worksheet.update(rowcol_to_a1(1, len(header) - len(missing) + 1), [missing])
    if updates:
        col_number = {col: i + 1 for i, col in enumerate(header)}
        worksheet.batch_update(
            [{'range': rowcol_to_a1(pos + 2, col_number[col]), 'values': [[val]]} for pos, col, val in updates],
            value_input_option='USER_ENTERED',
        )
    print(f'added {len(missing)} headers, wrote {len(updates)} cells')